# MedMNIST native-28 reproducibility — Batch 1 (verification)

**Protocol:** native 28x28, ResNet-18, 100 epochs, batch 128, Adam lr=0.001, MultiStepLR(50,75,γ=0.1), seeds 17/29/43/71/101. Upstream pinned at `70b6b3a`.

**Batch 1:** 15 runs (breastmnist, retinamnist, pneumoniamnist × 5 seeds), est. 4-6 T4 GPU-hours. Settings: **GPU ON, Internet ON**.

After the run: download `/kaggle/working/results_batch1.jsonl` and the `runs/official28/` tree, and commit results to the benchmark repo.

In [ ]:
!pip install -q medmnist tensorboardX scikit-learn
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no-gpu")

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("bench"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/habib-analyst/MedMNIST_v2_Reproducibility_Benchmark.git", "bench"], check=True)
os.chdir("/kaggle/working/bench")
subprocess.run(["git", "pull", "--ff-only"], check=False)
ext = "external/medmnist-experiments"
if not os.path.exists(ext):
    os.makedirs("external", exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/MedMNIST/experiments.git", ext], check=True)
subprocess.run(["git", "-C", ext, "fetch", "--depth", "1", "origin", "70b6b3a7ad7afddff1df2a3b735235830fbdb142"], check=True)
subprocess.run(["git", "-C", ext, "checkout", "70b6b3a7ad7afddff1df2a3b735235830fbdb142"], check=True)
print("setup done; upstream at", subprocess.run(["git", "-C", ext, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
import json, subprocess, sys, time
run_ids = ["2d-breastmnist-resnet18-28-seed17", "2d-breastmnist-resnet18-28-seed29", "2d-breastmnist-resnet18-28-seed43", "2d-breastmnist-resnet18-28-seed71", "2d-breastmnist-resnet18-28-seed101", "2d-retinamnist-resnet18-28-seed17", "2d-retinamnist-resnet18-28-seed29", "2d-retinamnist-resnet18-28-seed43", "2d-retinamnist-resnet18-28-seed71", "2d-retinamnist-resnet18-28-seed101", "2d-pneumoniamnist-resnet18-28-seed17", "2d-pneumoniamnist-resnet18-28-seed29", "2d-pneumoniamnist-resnet18-28-seed43", "2d-pneumoniamnist-resnet18-28-seed71", "2d-pneumoniamnist-resnet18-28-seed101"]
results = "/kaggle/working/results_batch1.jsonl"
open(results, "a").close()
t0 = time.time()
for i, rid in enumerate(run_ids, 1):
    print(f"\n===== [{i}/{len(run_ids)}] {rid} =====", flush=True)
    p = subprocess.run(
        [sys.executable, "scripts/run_official_2d_28.py", "--run_id", rid,
         "--gpu", "--results", results],
        cwd="/kaggle/working/bench", text=True)
    print(f"exit={p.returncode} elapsed_total={(time.time()-t0)/3600:.2f}h", flush=True)
    if p.returncode != 0:
        print("RUN FAILED -- continuing to next run")
print(f"\nBATCH DONE in {(time.time()-t0)/3600:.2f}h. Results: {results}")

In [ ]:
import json
recs = [json.loads(l) for l in open("/kaggle/working/results_batch1.jsonl") if l.strip()]
print(f"{len(recs)} records")
for r in recs:
    print(r["run_id"], "auc=", r.get("test_auc"), "acc=", r.get("test_acc"), "exit=", r.get("exit_code"), f"{r.get('elapsed_s',0)/60:.1f}min")